# Exercice 6 - Valorisation d'un Asset Swap

Bond 3Y, coupon 10%, prix marché 102.53%

Courbe swap : R(0,1)=5%, R(0,2)=6%, R(0,3)=7%

In [ ]:
import numpy as np

## Rappel

L'asset swap transforme les flux fixes d'une obligation en flux variables (Libor + spread). Le spread d'asset swap mesure la rémunération au dessus de Libor que reçoit l'investisseur.

$$s^{ASW} = \frac{P_{swap} - P_{marché}}{Annuité}$$

où $P_{swap}$ est le prix de l'obligation actualisé sur la courbe swap et l'annuité est la somme des discount factors.

In [ ]:
coupon = 0.10
prix = 102.53 / 100  # en pourcentage du nominal
T = 3

# taux ZC swap
taux = {1: 0.05, 2: 0.06, 3: 0.07}

In [ ]:
# facteurs d'actualisation
DF = {}
for t in range(1, T+1):
    DF[t] = 1 / (1 + taux[t])**t
    print(f"DF({t}) = {DF[t]:.6f}")

In [ ]:
# annuité
A = sum(DF[t] for t in range(1, T+1))
print(f"Annuité = {A:.4f}")

# prix theorique sur courbe swap
P_swap = sum(coupon * DF[t] for t in range(1, T+1)) + 1 * DF[T]
print(f"Prix swap = {P_swap*100:.4f}%")
print(f"Prix marché = {prix*100:.4f}%")

In [ ]:
# spread asset swap
s_asw = (P_swap - prix) / A
print(f"Spread ASW = {s_asw:.6f} = {s_asw*10000:.1f} bps")

## Analyse

Le spread est positif (~214 bps), ce qui signifie que le prix de marché est inférieur au prix théorique calculé sur la courbe swap. L'investisseur exige une compensation supérieure au taux interbancaire pour detenir ce bond, ce qui traduit le risque de crédit de l'emetteur.

En d'autres termes, l'obligataire reçoit Libor + 214 bps dans le swap, ce qui compense le fait que l'obligation est risquée (prix décoté par rapport au sans risque).

Note : le bond cote au dessus du pair (102.53%) car son coupon de 10% est très supérieur aux taux swap (5-7%). Mais une fois actualisé proprement sur la courbe swap, le prix théorique est encore plus élevé (~108%), la difference venant du credit spread.

In [ ]:
# pour mieux voir, calculons le YTM
from scipy.optimize import brentq

def prix_ytm(ytm):
    return sum(coupon/(1+ytm)**t for t in range(1, T+1)) + 1/(1+ytm)**T

ytm = brentq(lambda y: prix_ytm(y) - prix, 0.01, 0.3)
print(f"YTM = {ytm*100:.2f}%")
print(f"Taux swap 3Y = {taux[3]*100:.0f}%")
print(f"Spread YTM vs swap ≈ {(ytm - taux[3])*10000:.0f} bps")